In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv(r"C:\Users\Harshit Dalve\OneDrive\datasets\archive.zip")

In [3]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df.isnull().sum()

Category    0
Message     0
dtype: int64

In [6]:
df['Message'].duplicated().sum()

np.int64(415)

In [8]:
df.shape

(5573, 2)

In [9]:
df = df.drop_duplicates(subset=['Message'])

In [12]:
df['Category'].unique()

array(['ham', 'spam', '{"mode":"full"'], dtype=object)

In [14]:
df = df[df["Category"].isin(["ham", "spam"])].copy()

In [18]:
unique_category = df['Category'].unique()
category_number = {}

In [19]:
i=0
for j in unique_category:
    category_number[j] = i
    i+=1
df['Category'] = df['Category'].map(category_number)

In [20]:
df.head()

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
df['Message'] = df['Message'].apply(lambda x : x.lower())

In [24]:
import string
def remove_punc(txt):
    return txt.translate(str.maketrans("","",string.punctuation))

In [25]:
df['Message'] = df['Message'].apply(remove_punc)

In [28]:
import re

def remove_url(txt):
    return re.sub(r'https?://\S+|www\.\S+', '', txt)
df['Message'] = df['Message'].apply(remove_url)

In [29]:
def remove_emoji(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new = new+i
    return new
df['Message'] = df['Message'].apply(remove_emoji)

In [42]:
from sklearn.model_selection import train_test_split

In [44]:
X_train, X_test, y_train, y_test = train_test_split(df['Message'], df['Category'], test_size=0.2, random_state=42, stratify=df['Category'])

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfid = TfidfVectorizer(ngram_range=(1,2), max_features=5000)

In [51]:
X_train_tfid = tfid.fit_transform(X_train)
X_test_tfid = tfid.transform(X_test)

In [52]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

In [53]:
models = {
    "Naive Bayes" : {
        "model" : MultinomialNB(), 
        "params" : {
            "alpha" : [0.1, 0.5, 1.0, 2.0]
        }
    },
    "Logistic Regression" : {
        "model" : LogisticRegression(),
        "params" : {
            "C" : [0.1, 1, 10],
            "solver" : ["liblinear"]
        }
    }, 
     "Linear SVM": {
        "model": LinearSVC(),
        "params": {
            "C": [0.1, 1, 10]
        }
    }
    
}

In [57]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
result = []
for name, config in models.items():
    print(f"\nRunning GridSearch for {name}...") 
    grid = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        scoring="f1",
        cv=5,
        n_jobs=-1
    )
    grid.fit(X_train_tfid, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_tfid)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label=1)
    recall = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    result.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "CV F1": grid.best_score_,
        "Test Accuracy": accuracy,
        "Test Precision": precision,
        "Test Recall": recall,
        "Test F1": f1
    })


Running GridSearch for Naive Bayes...

Running GridSearch for Logistic Regression...

Running GridSearch for Linear SVM...


In [59]:
results_df = pd.DataFrame(result)

print(results_df)

                 Model                   Best Parameters     CV F1  \
0          Naive Bayes                    {'alpha': 0.1}  0.932589   
1  Logistic Regression  {'C': 10, 'solver': 'liblinear'}  0.906740   
2           Linear SVM                         {'C': 10}  0.932678   

   Test Accuracy  Test Precision  Test Recall   Test F1  
0       0.977713        0.964602     0.851562  0.904564  
1       0.976744        0.981481     0.828125  0.898305  
2       0.977713        0.948718     0.867188  0.906122  


In [60]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [61]:
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [64]:
df.loc[2]['Message']

'free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005 text fa to 87121 to receive entry questionstd txt ratetcs apply 08452810075over18s'

In [66]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)
df['Message'] = df['Message'].apply(remove)

In [67]:
df.loc[2]['Message']

'free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s'

In [68]:
from sklearn.model_selection import train_test_split

In [69]:
X_train, X_test, y_train, y_test = train_test_split(df['Message'], df['Category'], test_size=0.2, random_state=42, stratify=df['Category'])

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfid = TfidfVectorizer(ngram_range=(1,2), max_features=5000)

In [74]:
X_train_tfid_s = tfid.fit_transform(X_train)
X_test_tfid_s = tfid.transform(X_test)

In [75]:
models = {
    "Naive Bayes" : {
        "model" : MultinomialNB(), 
        "params" : {
            "alpha" : [0.1, 0.5, 1.0, 2.0]
        }
    },
    "Logistic Regression" : {
        "model" : LogisticRegression(),
        "params" : {
            "C" : [0.1, 1, 10],
            "solver" : ["liblinear"]
        }
    }, 
     "Linear SVM": {
        "model": LinearSVC(),
        "params": {
            "C": [0.1, 1, 10]
        }
    }
    
}

In [76]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
result = []
for name, config in models.items():
    print(f"\nRunning GridSearch for {name}...") 
    grid = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        scoring="f1",
        cv=5,
        n_jobs=-1
    )
    grid.fit(X_train_tfid_s, y_train)
    best_model = grid.best_estimator_
    y_pred_s = best_model.predict(X_test_tfid_s)
    accuracy_s = accuracy_score(y_test, y_pred_s)
    precision_s = precision_score(y_test, y_pred_s, pos_label=1)
    recall_s = recall_score(y_test, y_pred_s, pos_label=1)
    f1_s = f1_score(y_test, y_pred_s, pos_label=1)
    result.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "CV F1": grid.best_score_,
        "Test Accuracy": accuracy_s,
        "Test Precision": precision_s,
        "Test Recall": recall_s,
        "Test F1": f1_s
    })


Running GridSearch for Naive Bayes...

Running GridSearch for Logistic Regression...

Running GridSearch for Linear SVM...


In [77]:
results_df_s = pd.DataFrame(result)

print(results_df_s)

                 Model                   Best Parameters     CV F1  \
0          Naive Bayes                    {'alpha': 0.1}  0.929959   
1  Logistic Regression  {'C': 10, 'solver': 'liblinear'}  0.878540   
2           Linear SVM                          {'C': 1}  0.901953   

   Test Accuracy  Test Precision  Test Recall   Test F1  
0       0.975775        0.947826     0.851562  0.897119  
1       0.972868        0.954545     0.820312  0.882353  
2       0.974806        0.955357     0.835938  0.891667  


In [78]:
print(df["Category"].value_counts())

Category
0    4516
1     641
Name: count, dtype: int64


In [79]:
print(df["Category"].value_counts(normalize=True))

Category
0    0.875703
1    0.124297
Name: proportion, dtype: float64


In [80]:
from sklearn.pipeline import Pipeline
models = {

    "Naive Bayes": {
        "model": MultinomialNB(),
        "params": {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [5000, 10000],
            "model__alpha": [0.1, 0.5, 1.0]
        }
    },

    "Logistic Regression": {
        "model": LogisticRegression(max_iter=2000),
        "params": {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [5000, 10000],
            "model__C": [0.1, 1, 10]
        }
    },

    "Linear SVM": {
        "model": LinearSVC(),
        "params": {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [5000, 10000],
            "model__C": [0.1, 1, 10]
        }
    }
}

In [81]:
results = []
best_models = {}

for name, config in models.items():

    print(f"\n{'='*50}")
    print(f"Running: {name}")
    print(f"{'='*50}")

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("model", config["model"])
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=config["params"],
        scoring="f1",
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "CV F1": grid.best_score_,
        "Test Accuracy": accuracy,
        "Test Precision": precision,
        "Test Recall": recall,
        "Test F1": f1
    })

    best_models[name] = best_model



Running: Naive Bayes

Running: Logistic Regression

Running: Linear SVM


In [82]:
results_df_p = pd.DataFrame(results)

print("\nFINAL RESULTS")
print(results_df_p)


FINAL RESULTS
                 Model                                    Best Parameters  \
0          Naive Bayes  {'model__alpha': 0.1, 'tfidf__max_features': 5...   
1  Logistic Regression  {'model__C': 10, 'tfidf__max_features': 10000,...   
2           Linear SVM  {'model__C': 1, 'tfidf__max_features': 10000, ...   

      CV F1  Test Accuracy  Test Precision  Test Recall   Test F1  
0  0.922129       0.976744        0.933333     0.875000  0.903226  
1  0.897789       0.971899        0.945946     0.820312  0.878661  
2  0.915897       0.975775        0.947826     0.851562  0.897119  


In [83]:
from sklearn.pipeline import Pipeline
models = {

    "Naive Bayes": {
        "model": MultinomialNB(),
        "params": {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [5000, 10000],
            "model__alpha": [0.1, 0.5, 1.0]
        }
    },

    "Logistic Regression": {
        "model": LogisticRegression(max_iter=2000),
        "params": {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [5000, 10000],
            "model__C": [0.1, 1, 10],
            "model__class_weight": [None, "balanced"]
        }
    },

    "Linear SVM": {
        "model": LinearSVC(),
        "params": {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [5000, 10000],
            "model__C": [0.1, 1, 10],
            "model__class_weight": [None, "balanced"]
        }
    }
}

In [84]:
results = []
best_models = {}

for name, config in models.items():

    print(f"\n{'='*50}")
    print(f"Running: {name}")
    print(f"{'='*50}")

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("model", config["model"])
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=config["params"],
        scoring="f1",
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "CV F1": grid.best_score_,
        "Test Accuracy": accuracy,
        "Test Precision": precision,
        "Test Recall": recall,
        "Test F1": f1
    })

    best_models[name] = best_model



Running: Naive Bayes

Running: Logistic Regression

Running: Linear SVM


In [85]:
results_df_b = pd.DataFrame(results)

print("\nFINAL RESULTS")
print(results_df_b)


FINAL RESULTS
                 Model                                    Best Parameters  \
0          Naive Bayes  {'model__alpha': 0.1, 'tfidf__max_features': 5...   
1  Logistic Regression  {'model__C': 10, 'model__class_weight': 'balan...   
2           Linear SVM  {'model__C': 1, 'model__class_weight': 'balanc...   

      CV F1  Test Accuracy  Test Precision  Test Recall   Test F1  
0  0.922129       0.976744        0.933333     0.875000  0.903226  
1  0.918853       0.974806        0.918033     0.875000  0.896000  
2  0.923311       0.972868        0.916667     0.859375  0.887097  


In [86]:
Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("model", MultinomialNB(alpha=0.1))
])

,steps,"[('tfidf', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [87]:
import joblib

final_model = grid.best_estimator_

joblib.dump(final_model, "spam_detector.pkl")

['spam_detector.pkl']

In [88]:
import joblib

model = joblib.load("spam_detector.pkl")

email = """
Congratulations! You have won $5000.
Click here to claim your prize!
"""

prediction = model.predict([email])

print(prediction)

[1]
